# SPArrOW pipeline - Xenium

In this series of notebooks, we will analyze a 10X Genomics [Xenium](https://www.10xgenomics.com/platforms/xenium) dataset using the SPArrOW pipeline, a flexible, modular and scalable analysis pipeline that includes tools for image processing, cell segmentation, transcript allocation and cell type annotation. We will also explore the Xenium dataset itself, introduce the SpatialData data format and how you can plot spatial data, and go over some additional analysis options, including segmentation-free approaches and downstream analysis steps.

When you make use of the SPArrOW pipeline tools, please cite [Pollaris et al. (2024)](https://www.biorxiv.org/content/biorxiv/early/2024/07/06/2024.07.04.601829.full.pdf) and the Harpy paper (accepted for publication in Bioinformatics).

## Import packages

In [ ]:
import os

import dask.array as da
import geopandas as gpd
import harpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import spatialdata as sd
import spatialdata_io
import spatialdata_plot
import squidpy as sq


## 1. Xenium Ovarian Cancer dataset
The dataset was made publically available by 10X Genomics: [FFPE Human Ovarian Cancer with 5K Human Pan Tissue and Pathways Panel plus 100 Custom Genes](https://www.10xgenomics.com/datasets/xenium-prime-ffpe-human-ovarian-cancer). 

A single-cell RNA sequencing dataset on a consecutive section was also made available: [Human Ovarian Cancer FFPE Single Cell Gene Expression Flex (Next GEM)](https://www.10xgenomics.com/datasets/17k-human-ovarian-cancer-scFFPE). Additionally, an analysis guide from 10X Genomics on how to use this scRNA-seq reference to annotate cell types in the Xenium dataset using Seurat can be found [here](https://www.10xgenomics.com/analysis-guides/xenium-cell-type-annotation).

A good first step is to have a look at the Xenium Analysis Summary from Xenium Onboard Analysis, which can be found [here](https://cf.10xgenomics.com/samples/xenium/3.0.0/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_analysis_summary.html). Next, we can interactively explore the data using [Xenium Explorer](https://www.10xgenomics.com/support/software/xenium-explorer/latest). To do this, we need to locate the experiment.xenium file in the Xenium Onboard Analysis output folder and make sure the software is installed. In the case of this public dataset, we can also just open the [web demo](https://xenium.10xgenomics.com/?image=s3%2F10x.files%2Fxenium%2Fpreview%2FXenium_Prime_Ovarian_Cancer_FFPE_boosted%2Fexperiment.xenium&layers=cell~image~feature&cell_bt=cell&cell_c=groups&cell_g=10&cell_o=0.8&ct_c=Viridis&ct_o=0.8&ct_s=1_1276&cell_v=filled&z=14&off=&mi_v=true&axes=false&highlight=true&nav=true&rotation=0&target=32400.03_19524.55&tt_off=false&zoom=0.0179&feature=icon&feature_psc=6&feature_ps=circles&feature_off=0_188~190_220~222_349~351_421~423_765~767_806~808_1285~1287_1594~1596_2866~2868_3406~3408_4149~4151_4204~4206_4391~4393_4678~4680_4687~4689_4710~4712_4739~4741_4824~4826_4847~4849_9475&bin=10&d_c=Inferno&d_o=0.8&ii_minmax=__&ii_colors=_&ii_gamma=_&ii_b=0_0&ii_c=1_1&ii_n=H%26E%20Image%09H%26E%20Annotated%20Image&ii_o=1_1&ii_p=s3%2F10x.files%2Fxenium%2Fpreview%2FXenium_Prime_Ovarian_Cancer_FFPE_boosted%2Fhe_image.ome.tif%09s3%2F10x.files%2Fxenium%2Fpreview%2FXenium_Prime_Ovarian_Cancer_FFPE_boosted%2Fhe_annotated_image.ome.tif&ii_t=kzyedFNXhj8hKR3b5KH0P4DrlUUPiIbAISkd2%2BSh9L%2BTPJ50U1eGP8PfKbFV3uJAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAPA%2F_kzyedFNXhj8hKR3b5KH0P4DrlUUPiIbAISkd2%2BSh9L%2BTPJ50U1eGP8PfKbFV3uJAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAPA%2F&ii_v=false_false).

More information about Xenium can be found here:
- [Xenium Analyzer](https://www.10xgenomics.com/instruments/xenium-analyzer)
- [Xenium Onboard Analysis](https://www.10xgenomics.com/support/software/xenium-onboard-analysis/latest)
- [Xenium Explorer](https://www.10xgenomics.com/support/software/xenium-explorer/latest)
- [Gene panels](https://www.10xgenomics.com/products/xenium-panels#overview)
- [Xenium Prime 5K (Tech note)](https://cdn.10xgenomics.com/image/upload/v1721239182/support-documents/CG000775_Prime5K_DataHighlightsTN_RevA_updated.pdf)
- [Multimodal Cell Segmentation (Tech note)](https://cdn.10xgenomics.com/image/upload/v1754601291/support-documents/CG000750_XeniumInSitu_CellSegmentation_TechNote_RevB.pdf).

In [ ]:
# NOTE to Arne: Can you have a look at which old scripts in the repo we can remove (to clean out the repo a bit). We probably want to check with Janick since she will still run some of the HPC stuff in her training.

In [ ]:
# NOTE to Arne: I'm not sure why, but my kernel kept dying in notebook 2. I think it was mainly because the full sdata.zarr was also loaded into memory, so I hope the students don't have the same issues.
# In any case, can you also have a look at whether the notebooks are reasonable to run for the student and that there aren't any major problems?

In [ ]:
# NOTE to Arne: The students need access to the following files:
#   The morphology_focus image: /data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs/morphology_focus/morphology_focus_0000.ome.tif
#   The original H&E: /data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_image.ome.tif
#   The warped H&E that I created: /data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/bigwarp/HE_warped_TPS.tiff
#   The 10X affine transformation matrix: /data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_imagealignment.csv

#   tumor region annotation: /data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/region_annotations/warped_HE/tumor.geojson
#   necrosis region annotation: /data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/region_annotations/warped_HE/necrosis.geojson
#   ovary region annotation: /data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/region_annotations/warped_HE/ovary.geojson
#   fallopian_tube region annotation: /data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/region_annotations/warped_HE/fallopian_tube.geojson
#   smooth_muscle region annotation: /data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/region_annotations/warped_HE/smooth_muscle.geojson

# They will need to know those files are in their file system to use them for BigWarp and Qupath.

# They will need the following software:
#   Harpy environment (including Napari, squidpy, cellpose 3.1.1.1)
#   Fiji + Bigwarp (they can download an installation with the the correct versions using this link: https://objectstor.vib.be/s00-spatial.catalyst-team/sw/fiji-bigwarp/fiji-win64-bigwarp-9.1.3.zip)
#   Qupath (I've used 0.6.0, but I'm pretty sure 0.5.1 should also be fine)

## 2. Create SpatialData
In the case of Xenium data, Harpy contains a convenient reader function that reads in the images, transcripts, segmentation masks and tables into a SpatialData object that is backed to a Zarr store. 

Note that Harpy has a similar reader function for Vizgen MERSCOPE datasets (`hp.io.merscope`) and that other imaging-based spatial transcriptomics datasets can be read using a combination of more general functions (`harpy.io.create_sdata`, `hp.io.read_transcripts`, `hp.im.add_image_layer`...). See [documentation Harpy](https://harpy.readthedocs.io/en/latest/index.html) for more details.

In [ ]:
# Specify paths
input_path = "/data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs"  # Set input path to the output directory of Xenium Onboard Analysis
output_path = "/data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/harpy"  # Set a path to save the output from Harpy

zarr_path = os.path.join(output_path, "sdata.zarr")
cropped_zarr_path = os.path.join(output_path, "cropped_sdata.zarr")

In [ ]:
# NOTE: Because the dataset is large, we will not read it in using the reader below, but we will download a cropped version from our registry.
# The code below is what we used to read in the data and crop it to the region of interest.

"""
# Read in Xenium data using Harpy's Xenium reader.
sdata = hp.io.xenium(
   path=input_path,
   to_coordinate_system="global",
   aligned_images=True,
   cells_labels=True,
   nucleus_labels=True,
   morphology_mip=False,
   morpholohy_focus=True,
   cells_table=True,
   filter_gene_names=['Unassigned', 'NegControl', 'DeprecatedCodeword', 'Intergenic'],
   output=zarr_path,
)

# We crop the SpatialData using a bounding box query
xmin = 10000 
xmax = 20000
ymin = 18000
ymax = 28000

cropped_sdata = sdata.query.bounding_box(
    axes=["x", "y"],
    min_coordinate=[xmin, ymin],
    max_coordinate=[xmax, ymax],
    target_coordinate_system="global",
)

# Next, we write the cropped SpatialData to disk
cropped_sdata = sd.deepcopy(cropped_sdata)
cropped_sdata.write(cropped_zarr_path)
"""

In [ ]:
sdata = sd.read_zarr(cropped_zarr_path)

## 3. Explore SpatialData
We can now explore the SpatialData object and the different layers contained within it. We will also have a look at some basic operations with SpatialData (reading, writing, adding, deleting, plotting, ...). More information about SpatialData can be found [here](https://spatialdata.scverse.org/en/stable/).

### 3.1 SpatialData

#### 3.1.1 Inspect contents in memory and on disk

In [ ]:
# Check SpatialData
sdata

In [ ]:
# Check if SpatialData is backed (i.e. Zarr store on disk)
print("SpatialData backed by a Zarr store:", sdata.is_backed())
print("Path to associated Zarr store:", sdata.path)

In [ ]:
# Check contents of Zarr store
print(f"Content of {sdata.path}:")
! ls -l {sdata.path} # On windows, use 'dir' instead of 'ls -l'

In [ ]:
# Check contents of images directory in Zarr store
images_dir = os.path.join(sdata.path, "images")
print(f"Content of {images_dir}:")
! ls -l {images_dir} # On windows, use 'dir' instead of 'ls -l'

#### 3.1.2 Read and write SpatialData

In [ ]:
# Write the SpatialData to Zarr (if not backed, but it should be backed by hp.io.xenium)
sdata.write(sdata.path)

# NOTE: Notice that it will not allow you to overwrite an existing Zarr store ("The Zarr store already exists.")

In [ ]:
# We can read in a preexisting Spatialdata (e.g. when coming back to an analysis if the SpatialData is no longer in memory)
sdata = sd.read_zarr(sdata.path)
sdata

#### 3.1.3 Subsetting, copying and removing SpatialData and spatial elements

In [ ]:
# We can crop the SpatialData using a bounding box query
xmin = 10000
xmax = 12000
ymin = 18000
ymax = 20000

test_sdata = sdata.query.bounding_box(
    axes=["x", "y"],
    min_coordinate=[xmin, ymin],
    max_coordinate=[xmax, ymax],
    target_coordinate_system="global",
)

test_sdata

In [ ]:
# We can check whether the cropped SpatialData is backed by a Zarr store
test_sdata.is_backed()

In [ ]:
# We can create a deepcopy of the SpatialData
test_sdata = sd.deepcopy(test_sdata)

# Next, we write the cropped SpatialData to disk
test_sdata_path = os.path.join(os.path.dirname(sdata.path), "test.zarr")
test_sdata.write(test_sdata_path)

In [ ]:
# To remove the cropped SpatialData from disk, we could simply navigate to the Zarr store and remove it from our file system. We can also remove it in the notebook itself using shutil.
import shutil

shutil.rmtree(test_sdata.path)  # Removes SpatialData from disk
del test_sdata  # Removes SpatialData from memory

In [ ]:
# We can also copy a spatial element
se = sd.deepcopy(sdata.images["morphology_focus_global"])
sdata["copy_of_morphology_focus_global"] = se

# We write the element to disk to make sure it is backed
sdata.write_element("copy_of_morphology_focus_global", overwrite=True)
sdata

In [ ]:
# To remove a spatial element, we can use `del`
del sdata.images[
    "copy_of_morphology_focus_global"
]  # This deletes the element from memory, but does not remove the corresponding directory in the Zarr store.
sdata

In [ ]:
# If we read in the cropped SpatialData, we can see the spatial element is still there
sdata = sd.read_zarr(sdata.path)
sdata

In [ ]:
# To remove an element completely, we should use both 'del' and 'delete_element_from_disk'
del sdata.images["copy_of_morphology_focus_global"]
sdata.delete_element_from_disk("copy_of_morphology_focus_global")

# Read back SpatialData
sdata = sd.read_zarr(sdata.path)
sdata

#### 3.1.4 Coordinate systems and transformations
All elements in a `SpatialData` object are assigned to one or more coordinate systems, which allows for storing multiple samples in the same `SpatialData` object (e.g. each in their own coordinate system). Transformations between spatial elements make sure they allow align with the coordinate systems.

Some common transformations are:
- Identity
- Translation
- Scale
- Sequence
- Affine
- ...

We refer to these tutorials for more information:
-  [Tutorial Harpy](https://harpy.readthedocs.io/en/latest/tutorials/advanced/coordinate_systems.html)
-  [Tutorial SpatialData](https://spatialdata.scverse.org/en/stable/tutorials/notebooks/notebooks/examples/transformations.html)

In [ ]:
# Inspect SpatialData to see coordinate systems
sdata

In [ ]:
# Inspect coordinate systems and transformations associated with specific spatial elements
from spatialdata.transformations import (
    Identity,
    get_transformation,
    remove_transformation,
    set_transformation,
)

get_transformation(sdata.images["morphology_focus_global"], get_all=True)

In [ ]:
# Let's create a new coordinate system
transformations = get_transformation(
    sdata.images["morphology_focus_global"], get_all=True
)
transformations["new_coordinate_system"] = Identity()
set_transformation(
    sdata["morphology_focus_global"],
    transformation=transformations,
    set_all=True,
    write_to_sdata=sdata,
)
sdata

In [ ]:
# We can now remove the coordinate system again
remove_transformation(
    sdata["morphology_focus_global"], to_coordinate_system="new_coordinate_system"
)
sdata

#### 3.1.5 Exploring Spatialdata interactively in Napari

In [ ]:
# Interactively plot SpatialData in Napari
from napari_spatialdata import Interactive

Interactive(sdata)

### 3.2 Layers
Images, Labels and Points are lazy if the `SpatialData` object is backed by a `.zarr` store. Lazy means they will not be 'pulled' into RAM, unless you ask for it (e.g. calling `.compute()`, `.persist()` on the Dask objects).

[Dask](https://www.dask.org/) enables out-of-core computation, allowing you to process datasets that exceed the available RAM, and also facilitates parallelized computations.

Note that currently Tables and Shapes are not lazy, and will be loaded into memory when you load a `SpatialData` object. In the future, shapes will probably also be lazy, https://github.com/scverse/spatialdata/issues/359.

Support for lazy Tables should also be coming soon in `SpatialData`, but note that there is limited `Dask` support in e.g. `Scanpy` https://scanpy-tutorials.readthedocs.io/en/latest/dask.html, which would mean Tables need to be pulled in memory when `Scanpy` functions are applied on it.

In [ ]:
# Let's return to our Spatial Data object and inspect which layers it contains
sdata

#### 3.2.1 Images

In [ ]:
# Inspect image layer
sdata.images[
    "morphology_focus_global"
]  # This is a multiscale image so it is an xarray.DataTree

In [ ]:
# Get spatial element using _get_spatial_element function
from harpy.image._image import _get_spatial_element

se = _get_spatial_element(
    sdata, layer="morphology_focus_global"
)  # Gets scale0 if it is a multiscale image
se  # This is an xarray.DataArray

In [ ]:
se.data  # This is a Dask array

In [ ]:
# Compute the image array
se.data.compute()  # This is a numpy array

In [ ]:
# Inspect image dimensions
print(
    "Image dimensions:", se.dims
)  # Image layers have c,(z),y,x dimension. z dimension is optional
print("Image shape:", se.shape)

In [ ]:
# Inspect data type
print(
    "Image dtype:", se.data.dtype
)  # The data type of an image layer can be integer or float: uint8, uint16, float32, ...

In [ ]:
# Inspect channel names
se.c.data

#### 3.2.2 Labels
Labels layers typically represent segmentation masks.

Labels and images are sometimes referred to as `raster` data.

In [ ]:
# Inspect labels layer
sdata.labels["nucleus_labels_global"]  # This is multiscale so it is a datatree.DataTree

In [ ]:
# Inspect data type
se = _get_spatial_element(
    sdata, layer="nucleus_labels_global"
)  # This is an xarray.DataArray
se.dtype  # Data type of labels is always int (i.e.: float is not allowed).

#### 3.2.3 Shapes
Shapes either represent the boundaries of a segmentation mask (e.g. cells), or region annotations (e.g. artifacts, tumor regions, ...). Shapes are of type GeoDataFrame. They can be manipulated using the [geopandas](https://geopandas.org/en/stable/) library. Shapes layers can be converted into labels layers by rasterization and labels layers can be converted into shapes layers by vectorization. When using `SPArrOW` to generate shapes (via e.g. `hp.im.segment` or `hp.sh.vectorize`), the index of the shapes layer holds the cell id (its name is 'cell_ID') and corresponds to the labels in the corresponding labels layer.

In [ ]:
# Convert labels layers into shapes layers
sdata = hp.sh.vectorize(
    sdata,
    labels_layer="nucleus_labels_global",
    output_layer="nucleus_shapes_global",
    overwrite=True,
)

sdata = hp.sh.vectorize(
    sdata,
    labels_layer="cell_labels_global",
    output_layer="cell_shapes_global",
    overwrite=True,
)

# NOTE: `spatialdata` also implements vectorization (`spatialdata.to_polygons`) and rasterization (`spatialdata.rasterize`), but these implementations are slower, and require much more RAM.

In [ ]:
# Inspect shapes layer
sdata.shapes["nucleus_shapes_global"]

In [ ]:
# Inspect data type
print(
    type(sdata.shapes["nucleus_shapes_global"])
)  # Shapes are of type GeoDataFrame and we can use geopandas to manipulate them.

In [ ]:
# Inspect index
sdata.shapes["nucleus_shapes_global"].index.name

In [ ]:
# Inspect geometry
sdata.shapes["nucleus_shapes_global"].geometry.head()

In [ ]:
# Quickly plot geometry of nucleus 29481
sdata.shapes["nucleus_shapes_global"].geometry[29481]

In [ ]:
# Convert shapes layer back to labels layer
se = _get_spatial_element(sdata, layer="nucleus_labels_global")

sdata = hp.im.rasterize(
    sdata,
    shapes_layer="nucleus_shapes_global",
    output_layer="nucleus_labels_global_copy",
    out_shape=se.shape,
    chunks=5000,
    overwrite=True,
)

<b>Excercise</b>:

- Are the labels layers `nucleus_labels_global` and `nucleus_labels_global_copy` equal? Do you expect them to be equal?

<details>
<summary>Click to reveal the solution</summary>

```python
se_1 = _get_spatial_element(sdata, layer="nucleus_labels_global")
se_2 = _get_spatial_element(sdata, layer="nucleus_labels_global_copy")

pixels_not_equal = (~ np.equal(se_2.data.compute(), se_1.data.compute())).sum() 

print(f"After roundtrip vectorization and rasterization, {pixels_not_equal} pixels are not equal.")

print("Number of cells in nucleus_labels_global:", len(da.unique(se_1.data).compute()))
print("Number of cells in nucleus_labels_global_copy:", len(da.unique(se_2.data).compute()))


<b>Excercise</b>:
- Remove the `nucleus_labels_global_copy` layer.

<details>
<summary>Click to reveal the solution</summary>

```python
del sdata.labels["nucleus_labels_global_copy"]
sdata.delete_element_from_disk("nucleus_labels_global_copy")

#### 3.2.4 Points
Points are DaskDataFrame object (see https://docs.dask.org/en/stable/dataframe.html). Points represent the spatial location of a feature. In our case this will almost always be a gene.

In [ ]:
# Inspect points layer
sdata.points["transcripts_global"]

In [ ]:
# Compute points layer as pandas dataframe
sdata.points["transcripts_global"].compute().head()  # This is a pandas.DataFrame

<b>Excercise</b>:
- Use the points layer `transcripts_global` to determine how many unique genes were measured.

<details>
<summary>Click to reveal the solution</summary>

```python
sdata.points["transcripts_global"]["gene"].nunique().compute()

#### 3.2.5 Tables
Tables are [AnnData](https://anndata.readthedocs.io/en/latest/) objects. An `AnnData` object (`adata`) contains following attributes:

- `adata.X`: Main data matrix (observations x features). Observations are generally cells, while features (or variables) are generally genes.
- `adata.obs`: Metadata for each observation (e.g. QC-metrics, cell types, morphological characteristics).
- `adata.var`: Metadata for each feature (e.g. QC-metrics, gene ontology).
- `adata.uns`: Unstructured information (e.g. color schemes, settings).
- `adata.obsm`: Embeddings or reduced dimensions (e.g., PCA or UMAP coordinates, spatial coordinates).

In [ ]:
# Inspect tables layer
sdata.tables["table_global"]

`adata.X`

- Data type is a sparse or dense NumPy array or SciPy sparse matrix.
- This is the core data matrix of the AnnData object, typically an m×n matrix, where m is the number of observations (cells), and n is the number of variables (genes).
- Stores the primary quantitative data for each cell/gene pair, such as raw counts, normalized expression values, or transformations of the raw quantifications.

In [ ]:
# Inspect count matrix
sdata.tables["table_global"].X

In [ ]:
# Get shape of data matrix
sdata.tables["table_global"].X.toarray().shape

In [ ]:
# Get matrix as dataframe
sdata.tables["table_global"].to_df().head()  # Not recommended for large datasets

`adata.obs`

- Data type is a `pandas.DataFrame`.
- Each row corresponds to an observation in adata.X.
- Each colum corresponds to metadata about each observation: QC-metrics, cell types, clustering IDs, sample IDs, morphological features...

In [ ]:
sdata.tables["table_global"].obs.head()

In [ ]:
# Inspect number of cells per segmentation method
sdata.tables["table_global"].obs["segmentation_method"].value_counts()

In [ ]:
# Inspect number of nuclei per cell
sdata.tables["table_global"].obs["nucleus_count"].value_counts()

In [ ]:
# Quick histogram of number of transcripts per cell
sdata.tables["table_global"].obs["transcript_counts"].hist(bins=50)

In [ ]:
# Prettier histogram of number of transcripts per cell using matplotlib
ax = (
    sdata.tables["table_global"]
    .obs["transcript_counts"]
    .plot.hist(bins=50, figsize=(6, 4), edgecolor="white", color="steelblue")
)

ax.set_xlabel("Transcripts per cell")
ax.set_ylabel("Number of cells")
ax.set_title("Distribution of transcript counts")
plt.tight_layout()


`adata.var`

- Data type is a `pandas.DataFrame`.
- Each row corresponds to a feature in adata.X.
- Each colum corresponds to metadata about each feature: QC-metrics, gene ontology...

In [ ]:
sdata.tables["table_global"].var.head()

`adata.uns`

- `.uns` (unstructured data) is a dictionary for storing additional, often unstructured, information relevant to the dataset.
- Data type is a dictionary where you can store various data types, such as strings, arrays, or even nested dictionaries.
- Typically used for storing global dataset information, annotations, and visualization settings, like color palettes for clusters or parameter settings for computational methods

In [ ]:
sdata.tables["table_global"].uns

`adata.obsm`

- `.obsm` is a mapping of additional multi-dimensional arrays associated with each observation.
- Data type is a dictionary-like structure where each entry is typically a matrix or array of coordinates.
- Stores embeddings, dimensional reductions, spatial coordinates, or other coordinate-based data associated with observations.

In [ ]:
sdata.tables["table_global"].obsm

In [ ]:
sdata.tables["table_global"].obsm["spatial"][
    :5
]  # Contains centroid coordinates of cells

`Region key` and `Instance key`
According to spatialdata, a table layer can be annotated by a spatial element (`labels`, `shapes`, `points`), but tables generated by `Harpy` will be annotated by a `labels` layer.

In [ ]:
from spatialdata.models import TableModel

sdata.tables["table_global"].uns[TableModel.ATTRS_KEY]

In [ ]:
from harpy.utils._keys import _INSTANCE_KEY, _REGION_KEY

print(
    "Instance key column:", _INSTANCE_KEY
)  # Column in .obs that will be used for cell_ID
print(
    "Region key column:", _REGION_KEY
)  # Column in .obs that will be used for linking AnnData object to spatial element (a labels layer, e.g. a segmentation mask).

In [ ]:
# Inspect instance and region keys in obs
sdata.tables["table_global"].obs.head()

In [ ]:
# Inspect instance key values in obs
sdata.tables["table_global"].obs[_INSTANCE_KEY].values[:5]

In [ ]:
# Shapes index correspond to cell_ID values
sdata.shapes["nucleus_shapes_global"].index[:5]

In [ ]:
# Segmentation labels correspond to cell_ID values
import dask.array as da

se = _get_spatial_element(sdata, layer="nucleus_labels_global")
da.unique(se.data).compute()[
    :6
]  # Note that there is an value of 0, but this is reserved for the background and is not found in the table layer or shapes layer.

<b>Excercise</b>:
- Explore the cell_area column in the table layer. What is the average? What does the distribution look like?

<details>
<summary>Click to reveal the solution</summary>

```python
sdata.tables["table_global"].obs['cell_area'].describe()

ax = sdata.tables["table_global"].obs['cell_area'].plot.hist(
    bins=50,
    figsize=(6,4),
    edgecolor='white',
    color='steelblue'
)

ax.set_xlabel("Cell area")
ax.set_ylabel("Number of cells")
ax.set_title("Distribution of cell areas")
plt.tight_layout()

### 3.3 Plotting
Plotting of the different spatial elements of a SpatialData object can be done using [spatialdata-plot](https://spatialdata.scverse.org/projects/plot/en/stable/plotting.html). However, in Harpy, we added a convenient wrapper around spatialdata-plot to allow easier cropping during plotting, consistent colors and a simpler API.

#### 3.3.1 spatialdata-plot

In [ ]:
# We can plot the DAPI image using spatialdata-plot
sdata.pl.render_images(
    element="morphology_focus_global",
    channel="DAPI",
    cmap="gray",
).pl.show(
    coordinate_systems="global",
    title="DAPI",
    colorbar=False,
)

In [ ]:
# To plot a crop of the image, we need to first do a bounding box query
min_x = 10000
max_x = 11000
min_y = 18000
max_y = 19000

sdata_crop = sdata.query.bounding_box(
    min_coordinate=[min_x, min_y],
    max_coordinate=[max_x, max_y],
    axes=("x", "y"),
    target_coordinate_system="global",
)

# We can plot the DAPI image using spatialdata-plot
sdata_crop.pl.render_images(
    element="morphology_focus_global",
    channel="DAPI",
    cmap="gray",
).pl.show(
    coordinate_systems="global",
    title="DAPI",
    colorbar=False,
)

In [ ]:
# To create more complex plots, we can chain the render functions
sdata_crop.pl.render_images(
    element="morphology_focus_global",
    channel="DAPI",
    cmap="gray",
).pl.render_labels(
    element="cell_labels_global", fill_alpha=0.3, contour_px=5, outline_alpha=0
).pl.render_shapes(
    element="nucleus_shapes_global",
    fill_alpha=0,
    outline_width=0.8,
    outline_alpha=1,
    outline_color="red",
).pl.show(
    coordinate_systems="global",
    title="DAPI with cell segmentation masks and nucleus shapes",
    colorbar=False,
)


In [ ]:
# To can also plot the genes and obs columns from the table layer
sdata_crop.pl.render_images(
    element="morphology_focus_global",
    channel="DAPI",
    cmap="gray",
).pl.render_labels(
    element="cell_labels_global",
    fill_alpha=0.3,
    contour_px=5,
    outline_alpha=0,
    cmap="rainbow",
    color="segmentation_method",
    table_name="table_global",
).pl.render_shapes(
    element="nucleus_shapes_global",
    fill_alpha=0,
    outline_width=0.8,
    outline_alpha=1,
    outline_color="red",
).pl.show(
    coordinate_systems="global",
    title="DAPI with cell segmentation masks and nucleus shapes",
    colorbar=False,
    legend_fontsize=10,
    legend_fontweight="bold",
    legend_loc="right margin",
    figsize=(5, 5),
)

In [ ]:
# However, we might want to clean up the plots a bit more, leading to large chunks of code.
min_x = 10000
max_x = 11000
min_y = 18000
max_y = 19000

sdata_crop = sdata.query.bounding_box(
    min_coordinate=[min_x, min_y],
    max_coordinate=[max_x, max_y],
    axes=("x", "y"),
    target_coordinate_system="global",
)

fig, ax = plt.subplots(
    figsize=(5, 5)
)  # We create a figure and axis object to have more control over the plot aesthetics.
# fig (Figure) refers to the entire canvas. It can contain one or many Axes (subplots). It controls global properties such as overall size, layout, background, and saving the final image.
# ax (Axes) corresponds to the actual plotting area inside the figure. It contains the x‑axis, y‑axis, titles, points, etc. Each subplot is one Axes object.

title = "DAPI with cell segmentation masks and nucleus shapes"

sdata_crop.pl.render_images(
    element="morphology_focus_global",
    channel="DAPI",
    cmap="gray",
).pl.render_labels(
    element="cell_labels_global",
    fill_alpha=0.3,
    contour_px=1,
    outline_alpha=0,
    cmap="rainbow",
    color="segmentation_method",
    table_name="table_global",
).pl.render_shapes(
    element="nucleus_shapes_global",
    fill_alpha=0,
    outline_width=0.8,
    outline_alpha=1,
    outline_color="red",
).pl.show(
    coordinate_systems="global",
    title=title,
    colorbar=False,
    figsize=(5, 5),
    ax=ax,
    legend_fontsize=10,
    legend_fontweight="bold",
    legend_loc="right margin",
    na_in_legend=True,
)

ax.set_title(title, fontsize=12)  # This allows us to set the title font size
ax.set_xlabel("x", fontsize=12)  # This adds a label to x-axis and sets the font size
ax.set_ylabel("y", fontsize=12)  # This adds a label to y-axis and sets the font size
ax.tick_params(
    axis="both", which="both", labelsize=10
)  # This allows us to set the tick label size

ax.set_xlim(
    min_x, max_x
)  # This limits the axes to the same bounding box as the query, which avoids shapes from a shapes layer breaking the frame.
ax.set_ylim(min_y, max_y)
ax.invert_yaxis()  # Setting the xlim and ylim inverts the axes and this line inverts the yaxis back to the correct orientation for microscopy images.

save_path = os.path.join(output_path, "test_plot.png")
fig.savefig(
    save_path,
    bbox_inches="tight",
    pad_inches=0.05,
    facecolor="white",
    transparent=False,
)  # This saves the plot to disk

# plt.close(fig) # This closes the figure so it is not displayed in the notebook.

print("Plot saved at:", save_path, flush=True)

#### 3.3.2 Harpy
Harpy has 2 main plotting functions to plot the contents of a SpatialData object: hp.pl.plot_sdata() and hp.pl.plot_sdata_gene(). Note that there are also other plotting functions for more specific use-cases, but we will not look into those here.

In [ ]:
# Plot DAPI image using hp.pl.plot_sdata()
ax = hp.pl.plot_sdata(
    sdata,
    img_layer="morphology_focus_global",
    channel="DAPI",
    crd=None,  # Define crop to plot (xmin, xmax, ymin, ymax)
    to_coordinate_system="global",
    render_images_kwargs={
        "cmap": "grey"
    },  # keyword arguments passed on to spatialdata-plot's .pl.render_images()
    show_kwargs={
        "title": "DAPI",
        "colorbar": False,
    },  # keyword arguments passed on to spatialdata-plot's .pl.show()
)

In [ ]:
# Plot all channels for a crop of the image
pixel_size = 0.2125  # We need to specify the coordinates in micron if we want to use the micron coordinate system

min_x = 10000 * pixel_size
max_x = 11000 * pixel_size
min_y = 18000 * pixel_size
max_y = 19000 * pixel_size

channels = ["DAPI", "ATP1A1/CD45/E-Cadherin", "18S", "AlphaSMA/Vimentin"]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, channel in zip(axes.ravel(), channels):
    ax = hp.pl.plot_sdata(
        sdata,
        img_layer="morphology_focus_global",
        channel=channel,
        crd=(
            min_x,
            max_x,
            min_y,
            max_y,
        ),  # Define crop to plot (xmin, xmax, ymin, ymax)
        to_coordinate_system="global_micron",  # We are plotting in the micron coordinates
        render_images_kwargs={"cmap": "grey"},
        show_kwargs={"title": channel, "colorbar": False},
        ax=ax,
    )

plt.tight_layout()
plt.show()

In [ ]:
# Plot cell segmentation mask
hp.pl.plot_sdata(
    sdata,
    img_layer="morphology_focus_global",
    channel="DAPI",
    labels_layer="cell_labels_global",
    crd=(10000, 11000, 18000, 19000),
    render_images_kwargs={"cmap": "grey"},
    render_labels_kwargs={"fill_alpha": 0.4, "contour_px": 3, "outline_alpha": 1},
    show_kwargs={"title": "Cell segmentation", "colorbar": False},
)

In [ ]:
# Plot transcripts for a specific gene
hp.pl.plot_sdata_genes(
    sdata,
    img_layer="morphology_focus_global",
    channel="DAPI",
    points_layer="transcripts_global",
    crd=(10000, 11000, 18000, 19000),
    color="cornflowerblue",
    genes=["PLXNB1"],  # List of genes to plot (if None, all genes will be plotted)
    frac=1,  # We can also randomly subsampling transcripts to speed up plotting
    size=1,
    render_images_kwargs={"cmap": "grey"},
    show_kwargs={"title": "Transcripts", "colorbar": False},
    to_coordinate_system="global",
)

## 4. BigWarp registration

### 4.1 Registration using BigWarp
Since this dataset contains an H&E that was acquired after the Xenium run, we will register it to the fluorescent images and transcript coordinate space using the [BigWarp-plugin](https://imagej.net/plugins/bigwarp) in [FIJI](https://fiji.sc/) and add it to the SpatialData object.

For this notebook, we will follow the instructions as outlined in this [documentation](https://github.com/vibspatial/bigwarp). Specifically, we will download an installation of FIJI and BigWarp using the link in the documentation, create a BigWarp dataset, place landmarks on the DAPI (fixed image) and H&E images (moving image), and then export a warped version of the H&E image using a Thin-Plate Spline transformation. Note that we will not create the conda environment as described in the documentation since we will export to TIFF (and not to Zarr).

### 4.2 Adding the warped H&E image to the SpatialData object

In [ ]:
# Specify the path to the warped H&E image
path_image = "/data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/bigwarp/HE_warped_TPS.tiff"

# The warped H&E image is read using dask image
from dask_image.imread import imread

img = imread(path_image)

img = img.squeeze().transpose((2, 0, 1))  # Remove unused dimensions

sdata = hp.im.add_image_layer(
    sdata,  # The SpatialData object to which the new image layer will be added.
    arr=img,  # The array containing the image data to be added.
    dims=("c", "y", "x"),  # A tuple specifying the dimensions of the image data
    output_layer="he_image_bigwarp",  # The name of the output layer where the image data will be stored.
    chunks=1024,
    overwrite=True,
    scale_factors=(2, 2, 2),
)

In [ ]:
# NOTE to Arne: I noticed with the global_micron coordinate system (which I still think is a good idea), you have to be pretty intentional to make sure all operations you do
# (I think mainly adding images and region annotations) end up in that coordinate system as well. I decided not to bother for the training, but maybe we should think a bit about how we make this easier.

In [ ]:
# NOTE to Arne: Adding two large H&E images is probably a bit much. I think they are around 10GB each. Can we expect students to struggle with this performance-wise? Or is it mostly disk space that could be the issue?

In [ ]:
# Plot warped H&E
sdata.pl.render_images(
    "he_image_bigwarp"
).pl.show()  # spatialData-plot knows to plot an H&E as an RGB image.

In [ ]:
# Let's check whether the H&E is registered correctly to other spatial elements
min_x = 10000
max_x = 11000
min_y = 18000
max_y = 19000

sdata_crop = sdata.query.bounding_box(
    min_coordinate=[min_x, min_y],
    max_coordinate=[max_x, max_y],
    axes=("x", "y"),
    target_coordinate_system="global",
)

fig, ax = plt.subplots(figsize=(5, 5))

sdata_crop.pl.render_images(
    element="he_image_bigwarp",
).pl.render_shapes(
    element="nucleus_shapes_global",
    fill_alpha=0,
    outline_width=0.6,
    outline_alpha=1,
    outline_color="lightblue",
).pl.show(
    coordinate_systems="global",
    colorbar=False,
    title="BigWarp H&E with nucleus shapes",
    ax=ax,
)

ax.set_xlabel("x")
ax.set_ylabel("y")

ax.set_xlim(min_x, max_x)
ax.set_ylim(min_y, max_y)
ax.invert_yaxis()

### 4.3 Adding the original H&E image with an affine transformation
Alternatively, we can use the affine transformation matrix supplied by 10X Genomics and add the original H&E with the affine transformation specified in the SpatialData object. This workflow can be useful if you want to use the affine transformation matrix from BigWarp or some other registration method. Note that nonlinear transformations (e.g. TPS from BigWarp) are not supported by SpatialData.

In [ ]:
from spatialdata.transformations import Affine

# Specify the path to the original H&E image
path_image = "/data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_image.ome.tif"
path_transformation = "/data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_imagealignment.csv"

# Read in the transformation parameters from the CSV file
transformations_df = pd.read_csv(
    path_transformation, header=None
)  # We read in the affine transformation supplied by 10X Genomics as a pandas dataframe.
A = transformations_df.values  # Convert to a numpy array
affine_transform = Affine(A, input_axes=("x", "y"), output_axes=("x", "y"))

# The original H&E image is read using dask image
img = imread(path_image)
img = img.squeeze().transpose((2, 0, 1))  # Remove unused dimensions

transformations = {"global": affine_transform}

sdata = hp.im.add_image_layer(
    sdata,
    arr=img,
    dims=("c", "y", "x"),
    output_layer="he_image_original",
    chunks=1024,
    overwrite=True,
    transformations=transformations,  # We can specify the transformations to be added to the image layer here
    scale_factors=(2, 2, 2),
)

In [ ]:
# Let's check whether the H&E is registered correctly to other spatial elements
min_x = 10000
max_x = 11000
min_y = 18000
max_y = 19000

sdata_crop = sdata.query.bounding_box(
    min_coordinate=[min_x, min_y],
    max_coordinate=[max_x, max_y],
    axes=("x", "y"),
    target_coordinate_system="global",
)

fig, ax = plt.subplots(figsize=(5, 5))

sdata_crop.pl.render_images(
    element="he_image_original",
).pl.render_shapes(
    element="nucleus_shapes_global",
    fill_alpha=0,
    outline_width=0.6,
    outline_alpha=1,
    outline_color="lightblue",
).pl.show(
    coordinate_systems="global",
    colorbar=False,
    title="Original H&E (affine transformation) with nucleus shapes",
    ax=ax,
)

ax.set_xlabel("x")
ax.set_ylabel("y")

ax.set_xlim(min_x, max_x)
ax.set_ylim(min_y, max_y)
ax.invert_yaxis()

## 5. Region annotation
We will now create some region annotations on the warped H&E image. This makes sure the region annotations are in the same coordinate space as the fluorescent images and the transcripts. This is by far the easiest and most straight-forward workflow and is usually recommended. We can do this both in Napari and in Qupath.

However, in some cases, you may prefer to annotate on the original unwarped H&E and transform the region annotations according to the BigWarp tranformation. For this more complex scenario, we have created some documentation and tools, which you can find [here](https://github.com/vibspatial/bigwarp/blob/main/transform_geometries/transform_geometries.ipynb).

Note that you also have the option to annotate on the original unwarped H&E and add the region annotations to the SpatialData object with an affine transformation matrix specified (see 5.3). This, of course, requires that you know the affine transformation matrix. Since this is very similar to what was covered in 5.3, we will not cover it here again.

### 5.1 Region annotation in Napari
In napari, we will create a new shapes layer (rename the layer to 'region_annotation'), annotate a region of interest, save to sdata using Shift + E and close Napari.

For more info, see: https://spatialdata.scverse.org/en/latest/tutorials/notebooks/notebooks/examples/napari_rois.html. Note that there is, currently, only support for saving rectangles, polygons and points.

In [ ]:
# Open Napari to create region annotation
Interactive(sdata)

In [ ]:
# Let's check the sdata object to see whether the layer was correctly added
sdata

In [ ]:
# We need to make sure the shapes layer is backed to zarr
from spatialdata import read_zarr

sdata.write_element(element_name="region_annotation")
sdata = read_zarr(sdata.path)
sdata

In [ ]:
# Let's plot the region annotation
sdata.pl.render_images(element="he_image_bigwarp").pl.render_shapes(
    element="region_annotation",
    color="magenta",
    fill_alpha=0.15,
    outline_alpha=1,
    outline_color="magenta",
).pl.show(
    coordinate_systems="global",
    title="Region annotation Napari",
)

In [ ]:
# We can export the annotation as a GeoJSON to, for example, read it into Xenium Explorer
sdata.shapes["region_annotation"].to_file(
    os.path.join(output_path, "region_annotation.geojson"), driver="GeoJSON"
)

<b>Excercise</b>:
- Create your own region annotation in Napari, add it to the SpatialData and plot the results.

### 5.2 Qupath region annotation
Qupath is open source software for bioimage analysis. It has some really nice tools for annotating regions of interest and artifacts in images. We will use these for our dataset and add the results to the SpatialData object.

1. If necessary, install QuPath (v.0.6.0) using this [link](https://qupath.github.io/).
2. Open Qupath and click `Create project` in the top left corner.
3. Create a new empty folder for the project and click `Select Folder`.
4. To add images, go to `Add images` and provide the path to the warped H&E image.
5. Explore the image by going to `Brightness & contrast`, select the channels you want to visualize and adjust the settings to improve the visualization.
6. Annotate a region using the annotation tools.
7. Export the region annotation. Select all artifact annotations you want to export and go to `File`, `Export objects as GeoJSON` and set to `Selected objects` and `Compression` to None. Click `OK` and pick a file name.

In [ ]:
# We will add the region annotations we already created based on the annotations supplied by 10X Genomics.
import geopandas as gpd

paths_to_GeoJSONs = {
    "tumor": "/data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/region_annotations/warped_HE/tumor.geojson",
    "necrosis": "/data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/region_annotations/warped_HE/necrosis.geojson",
    "ovary": "/data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/region_annotations/warped_HE/ovary.geojson",
    "fallopian_tube": "/data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/region_annotations/warped_HE/fallopian_tube.geojson",
    "smooth_muscle": "/data/groups/technologies/spatial.catalyst/Training/training_data/Xenium_Ovarian_Cancer_FFPE/analysis/region_annotations/warped_HE/smooth_muscle.geojson",
}

for name, path in paths_to_GeoJSONs.items():
    gdf_regions = gpd.read_file(path)
    sdata = hp.sh.add_shapes_layer(
        sdata, input=gdf_regions, output_layer=name, overwrite=True
    )

In [ ]:
# Let's check whether the region annotations were added correctly
sdata.pl.render_images(element="he_image_bigwarp").pl.render_shapes(
    element="tumor", color="red", fill_alpha=0.15, outline_alpha=1, outline_color="red"
).pl.render_shapes(
    element="necrosis",
    color="black",
    fill_alpha=0.15,
    outline_alpha=1,
    outline_color="black",
).pl.render_shapes(
    element="ovary",
    color="lightgreen",
    fill_alpha=0.15,
    outline_alpha=1,
    outline_color="lightgreen",
).pl.render_shapes(
    element="smooth_muscle",
    color="blue",
    fill_alpha=0.15,
    outline_alpha=1,
    outline_color="blue",
).pl.render_shapes(
    element="fallopian_tube",
    color="darkgreen",
    fill_alpha=0.15,
    outline_alpha=1,
    outline_color="darkgreen",
).pl.show(
    coordinate_systems="global",
    title="Region annotations Qupath",
)

<b>Excercise</b>:
- Create your own region annotation in Qupath and add it to the SpatialData. Plot the resulting shapes layer.

### 5.3 ADVANCED: Registering region annotations using BigWarp
Follow the steps detailed in this [documentation](https://github.com/vibspatial/bigwarp/blob/main/transform_geometries/transform_geometries.ipynb). To sucessfully run this, you need to download the `vibspatial/bigwarp` github repository, open the `transform_geometries.ipynb` notebook, adjust the paths and follow the instructions.

In this training, we will register one of the region annotations created in Qupath on the warped H&E image (since we already have those) and transform it in the reverse direction to align it with the original H&E. We can check whether we were successful by opening both the original H&E and the warped region annotation GeoJSON in Qupath.